# Ejercicio 4 - Analisis exploratorio NYC TLC 2026 (DuckDB)

Fuente: `/workspace/data/raw/yellow/2026/*.parquet` y `/workspace/data/raw/green/2026/*.parquet` (enero-agosto 2026). Las consultas son las mismas de `sql/04_eda.sql`; la explicacion completa esta en `docs/04_analisis_exploratorio.md`.

## Preparacion: vista unificada `viajes`

Yellow usa `tpep_*` y green `lpep_*`; se normalizan a `pickup`/`dropoff` con `union_by_name` y se agrega `servicio`, `filename` y `duracion_min`.

In [1]:
import duckdb, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, os
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 100)
os.makedirs('/workspace/docs/img', exist_ok=True)
con = duckdb.connect()
con.execute("SET memory_limit='2GB'; SET threads=4; SET temp_directory='/tmp/duck_tmp'")
con.execute("""CREATE OR REPLACE VIEW viajes AS
WITH y AS (
    SELECT 'yellow' AS servicio, tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff,
           passenger_count, trip_distance, PULocationID, DOLocationID, payment_type,
           fare_amount, extra, mta_tax, tip_amount, tolls_amount, total_amount,
           CAST(NULL AS BIGINT) AS trip_type, filename
    FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet', union_by_name = true, filename = true)
), g AS (
    SELECT 'green' AS servicio, lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff,
           passenger_count, trip_distance, PULocationID, DOLocationID, payment_type,
           fare_amount, extra, mta_tax, tip_amount, tolls_amount, total_amount,
           trip_type, filename
    FROM read_parquet('/workspace/data/raw/green/2026/*.parquet', union_by_name = true, filename = true)
)
SELECT *, date_diff('second', pickup, dropoff) / 60.0 AS duracion_min
FROM (SELECT * FROM y UNION ALL BY NAME SELECT * FROM g)""")
con.sql('SELECT servicio, count(*) AS viajes FROM viajes GROUP BY servicio').df()

,servicio,viajes
0,yellow,29703355
1,green,337114


## 4.2 Comportamiento temporal

### Q1 - Volumen total y rango de fechas por servicio

In [2]:
q1 = con.sql("""
SELECT servicio, count(*) AS viajes, min(pickup) AS primer_pickup, max(pickup) AS ultimo_pickup,
       round(sum(total_amount)/1e6, 1) AS ingresos_millones_usd
FROM viajes GROUP BY servicio ORDER BY servicio
""").df()
q1

,servicio,viajes,primer_pickup,ultimo_pickup,ingresos_millones_usd
0,green,337114,2008-12-31 17:35:31,2026-08-31 23:58:28,8.6
1,yellow,29703355,2001-01-01 09:23:58,2026-08-31 23:59:59,893.2


### Q2 - Viajes por mes (pickup) y servicio, solo 2026

In [3]:
q2 = con.sql("""
SELECT month(pickup) AS mes, servicio, count(*) AS viajes
FROM viajes WHERE year(pickup) = 2026
GROUP BY ALL ORDER BY mes, servicio
""").df()
q2

,mes,servicio,viajes
0,1,green,40258
1,1,yellow,3724894
2,2,green,37388
3,2,yellow,3399866
4,3,green,44203
5,3,yellow,3952443
6,4,green,44243
7,4,yellow,3831256
8,5,green,44925
9,5,yellow,4090824


### Q3 - Viajes por dia de la semana (1=lunes ... 7=domingo)

In [4]:
q3 = con.sql("""
SELECT isodow(pickup) AS dia_semana, dayname(pickup) AS dia, servicio, count(*) AS viajes
FROM viajes WHERE year(pickup) = 2026
GROUP BY ALL ORDER BY dia_semana, servicio
""").df()
q3

,dia_semana,dia,servicio,viajes
0,1,Monday,green,47915
1,1,Monday,yellow,3552490
2,2,Tuesday,green,50803
3,2,Tuesday,yellow,4066818
4,3,Wednesday,green,52690
5,3,Wednesday,yellow,4339224
6,4,Thursday,green,55667
7,4,Thursday,yellow,4741507
8,5,Friday,green,50681
9,5,Friday,yellow,4509044


### Q4 - Viajes por hora del dia, con porcentaje sobre el servicio

In [5]:
q4 = con.sql("""
SELECT hour(pickup) AS hora, servicio, count(*) AS viajes,
       round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY servicio), 2) AS pct
FROM viajes WHERE year(pickup) = 2026
GROUP BY hour(pickup), servicio ORDER BY hora, servicio
""").df()
q4

,hora,servicio,viajes,pct
0,0,green,5295,1.57
1,0,yellow,933599,3.14
2,1,green,3374,1.00
3,1,yellow,621771,2.09
4,2,green,2459,0.73
5,2,yellow,415196,1.40
6,3,green,2043,0.61
7,3,yellow,298876,1.01
8,4,green,2057,0.61
9,4,yellow,245038,0.82


In [6]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
p = q2.pivot(index='mes', columns='servicio', values='viajes')
(p['yellow']/1e6).plot.bar(ax=ax[0], color='#e0a800'); ax[0].set_title('Yellow: viajes por mes (millones)'); ax[0].set_xlabel('mes 2026')
p3 = q3.pivot(index='dia_semana', columns='servicio', values='viajes')
(p3['yellow']/1e6).plot.bar(ax=ax[1], color='#e0a800'); ax[1].set_title('Yellow: viajes por dia (1=lun)'); ax[1].set_xlabel('dia')
p4 = q4.pivot(index='hora', columns='servicio', values='pct')
p4.plot(ax=ax[2], color={'yellow':'#e0a800','green':'#2e8b57'}); ax[2].set_title('% de viajes por hora'); ax[2].set_ylabel('%')
plt.tight_layout(); plt.savefig('/workspace/docs/img/04_temporal.png', dpi=80); plt.show()

## 4.2 Caracteristicas de los viajes y distribuciones

### Q5 - Caracteristicas de los viajes: distancia, duracion y pasajeros (datos plausibles)

In [7]:
q5 = con.sql("""
SELECT servicio, count(*) AS viajes,
       round(avg(trip_distance), 2) AS dist_media_mi,
       round(median(trip_distance), 2) AS dist_mediana_mi,
       round(avg(duracion_min), 2) AS dur_media_min,
       round(median(duracion_min), 2) AS dur_mediana_min,
       round(avg(passenger_count) FILTER (WHERE passenger_count BETWEEN 1 AND 6), 2) AS pasajeros_medio,
       round(avg(trip_distance / (duracion_min / 60.0)), 1) AS vel_media_mph
FROM viajes
WHERE year(pickup) = 2026 AND trip_distance > 0 AND trip_distance < 100
  AND duracion_min BETWEEN 1 AND 360
GROUP BY servicio ORDER BY servicio
""").df()
q5

,servicio,viajes,dist_media_mi,dist_mediana_mi,dur_media_min,dur_mediana_min,pasajeros_medio,vel_media_mph
0,green,320029,3.38,2.16,17.42,13.42,1.32,11.4
1,yellow,28285050,3.53,1.94,17.74,14.13,1.25,10.9


### Q6 - Distribucion de pasajeros

In [8]:
q6 = con.sql("""
SELECT servicio, passenger_count AS pasajeros, count(*) AS viajes,
       round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY servicio), 2) AS pct
FROM viajes WHERE year(pickup) = 2026
GROUP BY servicio, passenger_count ORDER BY servicio, pasajeros
""").df()
q6

,servicio,pasajeros,viajes,pct
0,green,0,4527,1.34
1,green,1,238269,70.68
2,green,2,28686,8.51
3,green,3,3322,0.99
4,green,4,2787,0.83
5,green,5,6340,1.88
6,green,6,4295,1.27
7,green,7,36,0.01
8,green,8,32,0.01
9,green,9,31,0.01


### Q7 - Percentiles de distancia, duracion y total_amount (datos plausibles)

In [9]:
q7 = con.sql("""
SELECT servicio, 'distancia_mi' AS variable,
       round(quantile_cont(trip_distance, 0.05),2) AS p05, round(quantile_cont(trip_distance, 0.25),2) AS p25,
       round(quantile_cont(trip_distance, 0.50),2) AS p50, round(quantile_cont(trip_distance, 0.75),2) AS p75,
       round(quantile_cont(trip_distance, 0.95),2) AS p95, round(quantile_cont(trip_distance, 0.99),2) AS p99
FROM viajes WHERE year(pickup)=2026 AND trip_distance > 0 AND trip_distance < 100 GROUP BY servicio
UNION ALL
SELECT servicio, 'duracion_min',
       round(quantile_cont(duracion_min, 0.05),2), round(quantile_cont(duracion_min, 0.25),2),
       round(quantile_cont(duracion_min, 0.50),2), round(quantile_cont(duracion_min, 0.75),2),
       round(quantile_cont(duracion_min, 0.95),2), round(quantile_cont(duracion_min, 0.99),2)
FROM viajes WHERE year(pickup)=2026 AND duracion_min BETWEEN 1 AND 360 GROUP BY servicio
UNION ALL
SELECT servicio, 'total_amount_usd',
       round(quantile_cont(total_amount, 0.05),2), round(quantile_cont(total_amount, 0.25),2),
       round(quantile_cont(total_amount, 0.50),2), round(quantile_cont(total_amount, 0.75),2),
       round(quantile_cont(total_amount, 0.95),2), round(quantile_cont(total_amount, 0.99),2)
FROM viajes WHERE year(pickup)=2026 AND total_amount > 0 AND total_amount < 1000 GROUP BY servicio
ORDER BY variable, servicio
""").df()
q7

,servicio,variable,p05,p25,p50,p75,p95,p99
0,green,distancia_mi,0.62,1.33,2.14,3.77,10.66,17.80
1,yellow,distancia_mi,0.50,1.10,1.92,3.93,12.57,19.57
2,green,duracion_min,4.40,8.77,13.38,20.72,44.22,75.23
3,yellow,duracion_min,4.12,8.70,14.22,22.37,44.00,71.13
4,green,total_amount_usd,9.54,15.00,20.50,29.70,57.78,97.20
5,yellow,total_amount_usd,12.25,17.55,23.69,34.71,77.75,105.75


### Q8 - Histograma de distancia (bins de 1 milla hasta 20, resto en 20+)

In [10]:
q8 = con.sql("""
SELECT servicio, least(floor(trip_distance), 20)::INT AS milla_bin, count(*) AS viajes
FROM viajes WHERE year(pickup)=2026 AND trip_distance > 0 AND trip_distance < 100
GROUP BY ALL ORDER BY servicio, milla_bin
""").df()
q8

,servicio,milla_bin,viajes
0,green,0,45121
1,green,1,105221
2,green,2,64908
3,green,3,34371
4,green,4,15938
5,green,5,11401
6,green,6,11094
7,green,7,7665
8,green,8,5927
9,green,9,4381


### Q9 - Histograma de duracion (bins de 5 min hasta 90, resto en 90+)

In [11]:
q9 = con.sql("""
SELECT servicio, least(floor(duracion_min / 5) * 5, 90)::INT AS min_bin, count(*) AS viajes
FROM viajes WHERE year(pickup)=2026 AND duracion_min BETWEEN 0 AND 360
GROUP BY ALL ORDER BY servicio, min_bin
""").df()
q9

,servicio,min_bin,viajes
0,green,0,33619
1,green,5,80734
2,green,10,83047
3,green,15,51907
4,green,20,29653
5,green,25,17630
6,green,30,11293
7,green,35,7369
8,green,40,5192
9,green,45,3673


In [12]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for s, c in [('yellow', '#e0a800'), ('green', '#2e8b57')]:
    d = q8[q8.servicio == s]; ax[0].plot(d.milla_bin, 100*d.viajes/d.viajes.sum(), color=c, label=s)
    d = q9[q9.servicio == s]; ax[1].plot(d.min_bin, 100*d.viajes/d.viajes.sum(), color=c, label=s)
ax[0].set_title('Distancia (milla, 20 = 20+) % de viajes'); ax[1].set_title('Duracion (min, 90 = 90+) % de viajes')
ax[0].legend(); plt.tight_layout(); plt.savefig('/workspace/docs/img/04_distribuciones.png', dpi=80); plt.show()

## 4.3 Amarillo vs verde y pagos

### Q10 - Amarillo vs verde (comparacion de promedios de tarifa y propina, datos plausibles)

In [13]:
q10 = con.sql("""
SELECT servicio, count(*) AS viajes,
       round(avg(fare_amount), 2) AS tarifa_media,
       round(avg(tip_amount), 2) AS propina_media,
       round(avg(total_amount), 2) AS total_medio,
       round(avg(total_amount / trip_distance), 2) AS usd_por_milla,
       round(100.0 * avg((PULocationID <> DOLocationID)::INT), 1) AS pct_cruzan_zona
FROM viajes
WHERE year(pickup)=2026 AND trip_distance BETWEEN 0.1 AND 100 AND fare_amount > 0
  AND total_amount BETWEEN 1 AND 1000 AND duracion_min BETWEEN 1 AND 360
GROUP BY servicio ORDER BY servicio
""").df()
q10

,servicio,viajes,tarifa_media,propina_media,total_medio,usd_por_milla,pct_cruzan_zona
0,green,313845,16.98,2.65,25.37,10.55,91.9
1,yellow,27974591,21.21,2.89,30.17,13.83,96.5


### Q11 - Tipo de viaje en verde (1=calle, 2=despacho) y valores nulos

In [14]:
q11 = con.sql("""
SELECT coalesce(trip_type::VARCHAR, 'NULL') AS trip_type, count(*) AS viajes
FROM viajes WHERE servicio = 'green' GROUP BY ALL ORDER BY 1
""").df()
q11

,trip_type,viajes
0,1,273386
1,2,14951
2,NULL,48777


### Q12 - Metodo de pago (0=flex, 1=tarjeta, 2=efectivo, 3=sin cargo, 4=disputa, 5=desconocido, 6=anulado)

In [15]:
q12 = con.sql("""
SELECT servicio, coalesce(payment_type::VARCHAR, 'NULL') AS payment_type, count(*) AS viajes,
       round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY servicio), 2) AS pct,
       round(avg(total_amount), 2) AS total_medio
FROM viajes WHERE year(pickup)=2026
GROUP BY servicio, payment_type ORDER BY servicio, payment_type
""").df()
q12

,servicio,payment_type,viajes,pct,total_medio
0,green,1,219974,65.25,25.84
1,green,2,65913,19.55,21.25
2,green,3,1688,0.50,6.09
3,green,4,750,0.22,2.59
4,green,NULL,48775,14.47,30.67
5,yellow,0,7716688,25.98,32.57
6,yellow,1,18940999,63.77,30.18
7,yellow,2,2708025,9.12,25.17
8,yellow,3,98136,0.33,12.67
9,yellow,4,239488,0.81,3.47


### Q13 - Propinas por metodo de pago (solo viajes con tarifa positiva)

In [16]:
q13 = con.sql("""
SELECT servicio, payment_type,
       count(*) AS viajes,
       round(avg(tip_amount), 2) AS propina_media,
       round(100.0 * avg((tip_amount > 0)::INT), 1) AS pct_con_propina,
       round(100.0 * sum(tip_amount) / sum(fare_amount), 1) AS propina_pct_de_tarifa
FROM viajes
WHERE year(pickup)=2026 AND fare_amount > 0 AND payment_type IN (1, 2)
GROUP BY ALL ORDER BY servicio, payment_type
""").df()
q13

,servicio,payment_type,viajes,propina_media,pct_con_propina,propina_pct_de_tarifa
0,green,1,219836,3.84,90.4,20.9
1,green,2,65797,0.00,0.0,0.0
2,yellow,1,18937331,4.28,91.1,21.5
3,yellow,2,2662810,0.00,0.0,0.0


### Q14 - Porcentaje de propina (sobre tarifa) en pagos con tarjeta, por tramos

In [17]:
q14 = con.sql("""
SELECT servicio,
       CASE WHEN tip_amount = 0 THEN '0% (sin propina)'
            WHEN tip_amount/fare_amount < 0.10 THEN '<10%'
            WHEN tip_amount/fare_amount < 0.20 THEN '10-20%'
            WHEN tip_amount/fare_amount < 0.30 THEN '20-30%'
            ELSE '>=30%' END AS tramo_propina,
       count(*) AS viajes
FROM viajes
WHERE year(pickup)=2026 AND payment_type = 1 AND fare_amount > 0
GROUP BY ALL ORDER BY servicio, tramo_propina
""").df()
q14

,servicio,tramo_propina,viajes
0,green,0% (sin propina),20998
1,green,10-20%,29266
2,green,20-30%,118271
3,green,<10%,12556
4,green,>=30%,38745
5,yellow,0% (sin propina),1685958
6,yellow,10-20%,2454562
7,yellow,20-30%,7871453
8,yellow,<10%,848042
9,yellow,>=30%,6077316


### Q15 - Componentes de la tarifa promedio (viajes plausibles)

In [18]:
q15 = con.sql("""
SELECT servicio, round(avg(fare_amount),2) AS tarifa, round(avg(extra),2) AS extra,
       round(avg(mta_tax),2) AS mta_tax, round(avg(tip_amount),2) AS propina,
       round(avg(tolls_amount),2) AS peajes, round(avg(total_amount),2) AS total
FROM viajes
WHERE year(pickup)=2026 AND fare_amount > 0 AND total_amount BETWEEN 1 AND 1000
GROUP BY servicio ORDER BY servicio
""").df()
q15

,servicio,tarifa,extra,mta_tax,propina,peajes,total
0,green,17.36,0.84,0.55,2.67,0.29,25.59
1,yellow,21.52,1.13,0.49,2.85,0.54,30.40


### Q16 - Ingresos y total medio por mes

In [19]:
q16 = con.sql("""
SELECT month(pickup) AS mes, servicio, round(sum(total_amount)/1e6, 2) AS ingresos_millones,
       round(avg(total_amount), 2) AS total_medio
FROM viajes WHERE year(pickup)=2026 AND total_amount BETWEEN 0 AND 1000
GROUP BY ALL ORDER BY mes, servicio
""").df()
q16

,mes,servicio,ingresos_millones,total_medio
0,1,green,0.98,24.30
1,1,yellow,109.90,29.82
2,2,green,0.91,24.36
3,2,yellow,103.18,30.59
4,3,green,1.10,25.01
5,3,yellow,119.55,30.41
6,4,green,1.13,25.54
7,4,yellow,115.36,30.23
8,5,green,1.17,26.05
9,5,yellow,125.10,30.69


In [20]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
p = q12.pivot(index='payment_type', columns='servicio', values='pct').fillna(0)
p.plot.bar(ax=ax[0], color={'yellow':'#e0a800','green':'#2e8b57'}); ax[0].set_title('% de viajes por payment_type')
t = q14.pivot(index='tramo_propina', columns='servicio', values='viajes'); (100*t/t.sum()).plot.bar(ax=ax[1], color={'yellow':'#e0a800','green':'#2e8b57'})
ax[1].set_title('Pago con tarjeta: tramo de propina (% de viajes)'); plt.tight_layout(); plt.savefig('/workspace/docs/img/04_pagos.png', dpi=80); plt.show()

## 4.3 Valores atipicos e inconsistencias

### Q17 - Inconsistencias - conteo por regla de calidad (un viaje puede violar varias reglas)

In [21]:
q17 = con.sql("""
SELECT servicio, regla, count(*) AS viajes FROM (
  SELECT servicio, unnest(list_filter([
        CASE WHEN year(pickup) <> 2026 THEN 'pickup fuera de 2026' END,
        CASE WHEN dropoff < pickup THEN 'duracion negativa' END,
        CASE WHEN duracion_min = 0 THEN 'duracion cero' END,
        CASE WHEN duracion_min > 360 THEN 'duracion > 6 h' END,
        CASE WHEN trip_distance = 0 THEN 'distancia 0' END,
        CASE WHEN trip_distance > 100 THEN 'distancia > 100 mi' END,
        CASE WHEN trip_distance < 0 THEN 'distancia negativa' END,
        CASE WHEN passenger_count = 0 THEN 'pasajeros 0' END,
        CASE WHEN passenger_count > 6 THEN 'pasajeros > 6' END,
        CASE WHEN passenger_count IS NULL THEN 'pasajeros NULL' END,
        CASE WHEN fare_amount < 0 THEN 'tarifa negativa' END,
        CASE WHEN total_amount < 0 THEN 'total negativo' END,
        CASE WHEN total_amount = 0 THEN 'total cero' END,
        CASE WHEN tip_amount < 0 THEN 'propina negativa' END,
        CASE WHEN total_amount > 1000 THEN 'total > 1000 usd' END
      ], x -> x IS NOT NULL)) AS regla
  FROM viajes
) GROUP BY ALL ORDER BY servicio, viajes DESC
""").df()
q17

,servicio,regla,viajes
0,green,pasajeros NULL,48775
1,green,distancia 0,12212
2,green,pasajeros 0,4527
3,green,duracion > 6 h,1104
4,green,total negativo,1023
5,green,tarifa negativa,999
6,green,total cero,543
7,green,duracion cero,229
8,green,pasajeros > 6,99
9,green,distancia > 100 mi,72


### Q18 - Viajes con fechas fuera de 2026 (por anio-mes de pickup)

In [22]:
q18 = con.sql("""
SELECT servicio, strftime(pickup, '%Y-%m') AS anio_mes, count(*) AS viajes
FROM viajes WHERE year(pickup) <> 2026
GROUP BY ALL ORDER BY viajes DESC LIMIT 15
""").df()
q18

,servicio,anio_mes,viajes
0,yellow,2025-12,6
1,yellow,2009-01,6
2,green,2008-12,5
3,green,2009-01,5
4,yellow,2008-12,4
5,green,2025-12,4
6,yellow,2001-01,1


### Q19 - Extremos de distancia (top 10)

In [23]:
q19 = con.sql("""
SELECT servicio, pickup, round(trip_distance,1) AS dist_mi, round(duracion_min,1) AS dur_min,
       fare_amount, total_amount
FROM viajes WHERE year(pickup)=2026
ORDER BY trip_distance DESC LIMIT 10
""").df()
q19

,servicio,pickup,dist_mi,dur_min,fare_amount,total_amount
0,yellow,2026-02-20 11:40:00,328522.2,22.0,31.11,35.86
1,yellow,2026-07-29 22:02:00,318129.1,31.0,44.18,48.93
2,yellow,2026-07-21 20:32:00,317017.9,24.0,38.19,42.94
3,yellow,2026-06-07 23:13:00,311565.4,6.0,19.23,23.98
4,yellow,2026-05-08 03:14:00,307491.5,18.0,26.20,35.59
5,yellow,2026-02-27 17:12:00,298292.1,46.0,74.10,75.60
6,yellow,2026-03-09 14:28:00,288381.7,17.0,21.39,31.37
7,yellow,2026-05-17 00:50:00,283971.8,20.0,36.16,40.16
8,yellow,2026-04-30 23:16:00,281576.1,40.0,35.90,39.95
9,yellow,2026-07-28 08:47:00,270530.3,33.0,65.91,70.66


### Q20 - Velocidades imposibles (> 80 mph con distancia > 1 mi)

In [24]:
q20 = con.sql("""
SELECT servicio, count(*) AS viajes
FROM viajes
WHERE year(pickup)=2026 AND duracion_min > 0 AND trip_distance > 1
  AND trip_distance / (duracion_min/60.0) > 80
GROUP BY servicio ORDER BY servicio
""").df()
q20

,servicio,viajes
0,green,603
1,yellow,6836


### Q21 - Top 10 zonas de origen (PULocationID) y su participacion

In [25]:
q21 = con.sql("""
SELECT PULocationID, count(*) AS viajes, round(100.0*count(*)/sum(count(*)) OVER (), 2) AS pct
FROM viajes WHERE year(pickup)=2026
GROUP BY PULocationID ORDER BY viajes DESC LIMIT 10
""").df()
q21

,PULocationID,viajes,pct
0,237,1307217,4.35
1,161,1223034,4.07
2,132,1181731,3.93
3,236,1170710,3.90
4,186,903997,3.01
5,162,902104,3.00
6,230,852978,2.84
7,142,847996,2.82
8,79,789273,2.63
9,170,776596,2.59


### Q22 - Impacto de los atipicos sobre la distancia media

In [26]:
q22 = con.sql("""
SELECT servicio,
       round(avg(trip_distance), 2) AS dist_media_cruda,
       round(avg(trip_distance) FILTER (WHERE trip_distance > 0 AND trip_distance < 100), 2) AS dist_media_limpia,
       max(trip_distance) AS dist_maxima
FROM viajes WHERE year(pickup)=2026
GROUP BY servicio ORDER BY servicio
""").df()
q22

,servicio,dist_media_cruda,dist_media_limpia,dist_maxima
0,green,13.35,3.36,179830.92
1,yellow,5.55,3.51,328522.20


## 4.5 Conclusiones

Ver `docs/04_analisis_exploratorio.md` para los hallazgos principales con cifras y su interpretacion.